In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
os.makedirs('/content/drive/MyDrive/SVR Project/Stats Test', exist_ok=True)

In [ ]:
!pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 4.7 MB/s eta 0:00:00


In [ ]:
# ============================================================
# SVR Subsampling: Multi-Run Statistical Validation
# Runs both Spatial and Residual criteria 5 times each
# Reports mean ± std and Wilcoxon signed-rank test
# ============================================================

import numpy as np
import pandas as pd
import time
from sklearn.datasets import make_friedman1
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from skopt import BayesSearchCV
from scipy.stats import wilcoxon
import sys
import warnings
warnings.filterwarnings('ignore')

# ── Import your algorithm ──────────────────────────────────
sys.path.insert(0, '/content/drive/MyDrive/SVR Project/Library')
try:
    from svr_residual_subsample import SVRSubsampleOptimizer, SVRConfig
    SUBSAMPLE_AVAILABLE = True
except ImportError:
    print("WARNING: Could not import svr_residual_subsample")
    SUBSAMPLE_AVAILABLE = False

# ── Experiment configuration ───────────────────────────────
SAMPLE_SIZES  = [1000, 5000, 10000, 15000, 20000, 30000, 40000, 45000, 50000]
SEEDS         = [42, 123, 456, 789, 1011]   # 5 independent runs
KERNEL_TYPE   = 'rbf'
KERNEL_PARAMS = {
    'rbf': {
        'C':       (1e-1, 1e+2, 'log-uniform'),
        'gamma':   (1e-4, 1e+0, 'log-uniform'),
        'epsilon': (1e-2, 1e+1, 'log-uniform'),
    }
}
TRAD_FIXED_SEED = 42   # Traditional SVR uses a fixed seed → run once per n


# ── Helper: Traditional SVR (run ONCE per dataset size) ───
def train_traditional_svr(X_train, y_train, X_test, y_test):
    """Deterministic because random_state is fixed."""
    svr = BayesSearchCV(
        SVR(kernel=KERNEL_TYPE),
        KERNEL_PARAMS[KERNEL_TYPE],
        cv=3,
        n_jobs=-1,
        n_points=5,
        n_iter=20,
        verbose=0,
        random_state=TRAD_FIXED_SEED,   # ← FIXED → deterministic
    )
    t0 = time.time()
    svr.fit(X_train, y_train)
    elapsed = time.time() - t0

    y_pred = svr.predict(X_test)
    return {
        'time_trad': elapsed,
        'r2_trad':   r2_score(y_test, y_pred),
        'mae_trad':  mean_absolute_error(y_test, y_pred),
        'rmse_trad': np.sqrt(mean_squared_error(y_test, y_pred)),
        'n_sv_trad': len(svr.best_estimator_.support_vectors_),
    }


# ── Helper: Subsample SVR (run 5 times with different seeds) ──
def train_subsample_svr(train_df, test_df, use_residual, random_state):
    """
    use_residual=True  → Residual criterion (this work)
    use_residual=False → Spatial criterion  (Camelo et al.)
    """
    # Dynamic fraction: at least 50 points or 1%, whichever is larger
    min_fraction = max(0.01, 50 / len(train_df))
    config = SVRConfig(
        subsample_fraction=min_fraction,
        r_set_fraction=0.1,
        num_neighbors=5,
        bayes_n_points=5,
        use_residual_criterion=use_residual,
        random_state=random_state,          # ← varies across runs
    )
    optimizer = SVRSubsampleOptimizer(config)

    t0 = time.time()
    model, info = optimizer.train(
        train_data=train_df,
        test_data=test_df,
        y_label_name='target',
        kernel_type=KERNEL_TYPE,
        kernel_params=KERNEL_PARAMS,
    )
    elapsed = time.time() - t0

    return {
        'time_sub': elapsed,
        'r2_sub':   info['final_metrics']['r2_score'],
        'mae_sub':  info['final_metrics']['mae'],
        'rmse_sub': info['final_metrics']['rmse'],
        'n_sv_sub': len(model.best_estimator_.support_vectors_),
    }


# ── Checkpoint helpers ────────────────────────────────────
TRAD_CHECKPOINT = '/content/drive/MyDrive/SVR Project/Stats Test/checkpoint_trad.csv'   # one row per completed n
RUNS_CHECKPOINT = '/content/drive/MyDrive/SVR Project/Stats Test/all_runs_raw.csv'      # one row per (n, seed, criterion)

def load_trad_checkpoint():
    """Returns dict {n: trad_metrics} for already-completed Traditional SVR runs."""
    import os
    if not os.path.exists(TRAD_CHECKPOINT):
        return {}
    df = pd.read_csv(TRAD_CHECKPOINT)
    result = {}
    for _, row in df.iterrows():
        n = int(row['n_samples'])
        result[n] = {
            'time_trad': row['time_trad'],
            'r2_trad':   row['r2_trad'],
            'mae_trad':  row['mae_trad'],
            'rmse_trad': row['rmse_trad'],
            'n_sv_trad': row['n_sv_trad'],
        }
    print(f"  📂 Loaded Traditional SVR checkpoint: {sorted(result.keys())}")
    return result

def save_trad_checkpoint(trad_done):
    """Save completed Traditional SVR results."""
    rows = [{'n_samples': n, **metrics} for n, metrics in trad_done.items()]
    pd.DataFrame(rows).to_csv(TRAD_CHECKPOINT, index=False)

def load_runs_checkpoint():
    """Returns (list_of_rows, set_of_completed_(n,seed,criterion) tuples)."""
    import os
    if not os.path.exists(RUNS_CHECKPOINT):
        return [], set()
    df = pd.read_csv(RUNS_CHECKPOINT)
    rows = df.to_dict('records')
    done = set(zip(df['n_samples'].astype(int),
                   df['seed'].astype(int),
                   df['criterion']))
    print(f"  📂 Loaded subsample checkpoint: {len(rows)} rows, "
          f"{len(done)} completed (n, seed, criterion) combos")
    return rows, done


# ── Main experiment loop ───────────────────────────────────
def run_experiment():

    # ── Load checkpoints ──────────────────────────────────
    trad_done = load_trad_checkpoint()   # {n: trad_metrics}
    all_rows, sub_done = load_runs_checkpoint()

    for n in SAMPLE_SIZES:
        print(f"\n{'='*60}")
        print(f"  n = {n:,}")
        print(f"{'='*60}")

        # Generate data with a FIXED seed so every run sees the same dataset
        X, y = make_friedman1(n_samples=n, n_features=20,
                              noise=1.0, random_state=42)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42)

        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s  = scaler.transform(X_test)

        # ── Traditional SVR — run ONCE, skip if already done ──
        if n in trad_done:
            trad = trad_done[n]
            print(f"  [Traditional SVR] ✅ loaded from checkpoint — "
                  f"time={trad['time_trad']:.1f}s  R²={trad['r2_trad']:.4f}")
        else:
            print(f"  [Traditional SVR] training (once)...")
            trad = train_traditional_svr(X_train_s, y_train, X_test_s, y_test)
            trad_done[n] = trad
            save_trad_checkpoint(trad_done)   # ← save immediately after trad finishes
            print(f"  [Traditional SVR] ✅ done — "
                  f"time={trad['time_trad']:.1f}s  R²={trad['r2_trad']:.4f}")

        # Build DataFrames for subsample functions
        cols = [f'feat_{i}' for i in range(20)]
        train_df = pd.DataFrame(X_train_s, columns=cols)
        train_df['target'] = y_train
        test_df  = pd.DataFrame(X_test_s,  columns=cols)
        test_df['target']  = y_test

        # ── 5 runs × 2 criteria ──
        for seed in SEEDS:
            for use_residual in [True, False]:
                criterion = 'Residual' if use_residual else 'Spatial'

                # Skip if already completed
                if (n, seed, criterion) in sub_done:
                    print(f"  [{criterion}] seed={seed} ✅ already done — skipping")
                    continue

                print(f"  [{criterion}] seed={seed} ...", end=' ', flush=True)

                try:
                    sub = train_subsample_svr(train_df, test_df,
                                              use_residual, seed)
                    speedup         = trad['time_trad'] / sub['time_sub']
                    r2_diff         = sub['r2_sub'] - trad['r2_trad']
                    mae_improvement = ((trad['mae_trad'] - sub['mae_sub'])
                                       / trad['mae_trad']) * 100
                    efficiency      = sub['r2_sub'] / sub['time_sub']

                    row = {
                        'kernel':          KERNEL_TYPE,
                        'n_samples':       n,
                        'seed':            seed,
                        'criterion':       criterion,
                        **trad,
                        **sub,
                        'speedup':         speedup,
                        'r2_diff':         r2_diff,
                        'mae_improvement': mae_improvement,
                        'efficiency':      efficiency,
                    }
                    all_rows.append(row)
                    sub_done.add((n, seed, criterion))
                    print(f"speedup={speedup:.1f}x  R²_sub={sub['r2_sub']:.4f}")

                    # Save after EVERY individual run
                    pd.DataFrame(all_rows).to_csv(RUNS_CHECKPOINT, index=False)

                except Exception as e:
                    print(f"ERROR: {e}")

        print(f"  ✅ n={n:,} complete — {len(all_rows)} total rows saved.")

    return pd.DataFrame(all_rows)


# ── Summary statistics ─────────────────────────────────────
def compute_summary(df):
    summary_rows = []
    for n in df['n_samples'].unique():
        for crit in ['Residual', 'Spatial']:
            sub = df[(df['n_samples'] == n) & (df['criterion'] == crit)]
            if sub.empty:
                continue
            # Traditional SVR values are identical across seeds
            row = {
                'n_samples':       n,
                'criterion':       crit,
                'time_trad_mean':  sub['time_trad'].mean(),
                'r2_trad_mean':    sub['r2_trad'].mean(),
                'rmse_trad_mean':  sub['rmse_trad'].mean(),
                'mae_trad_mean':   sub['mae_trad'].mean(),
                'n_sv_trad_mean':  sub['n_sv_trad'].mean(),
                'time_sub_mean':   sub['time_sub'].mean(),
                'time_sub_std':    sub['time_sub'].std(),
                'speedup_mean':    sub['speedup'].mean(),
                'speedup_std':     sub['speedup'].std(),
                'r2_sub_mean':     sub['r2_sub'].mean(),
                'r2_sub_std':      sub['r2_sub'].std(),
                'r2_diff_mean':    sub['r2_diff'].mean(),
                'r2_diff_std':     sub['r2_diff'].std(),
                'rmse_sub_mean':   sub['rmse_sub'].mean(),
                'rmse_sub_std':    sub['rmse_sub'].std(),
                'mae_sub_mean':    sub['mae_sub'].mean(),
                'mae_sub_std':     sub['mae_sub'].std(),
                'n_sv_sub_mean':   sub['n_sv_sub'].mean(),
                'n_sv_sub_std':    sub['n_sv_sub'].std(),
            }
            summary_rows.append(row)

    return pd.DataFrame(summary_rows).round(4)


# ── Wilcoxon signed-rank tests ─────────────────────────────
def run_wilcoxon_tests(df):
    print("\n" + "="*60)
    print("  WILCOXON SIGNED-RANK TESTS")
    print("="*60)

    results = {}

    # ── Test 1: Is Residual speedup > 1? ──
    res_speedups = df[df['criterion'] == 'Residual']['speedup'].values
    stat1, p1 = wilcoxon(res_speedups - 1, alternative='greater')
    results['residual_speedup_gt_1'] = {'statistic': stat1, 'p_value': p1}
    print(f"\n[Test 1] Residual speedup significantly > 1?")
    print(f"  W={stat1:.2f}, p={p1:.4f} → {'✅ YES (p<0.05)' if p1 < 0.05 else '❌ NO'}")

    # ── Test 2: Is Spatial speedup > 1? ──
    spa_speedups = df[df['criterion'] == 'Spatial']['speedup'].values
    stat2, p2 = wilcoxon(spa_speedups - 1, alternative='greater')
    results['spatial_speedup_gt_1'] = {'statistic': stat2, 'p_value': p2}
    print(f"\n[Test 2] Spatial speedup significantly > 1?")
    print(f"  W={stat2:.2f}, p={p2:.4f} → {'✅ YES (p<0.05)' if p2 < 0.05 else '❌ NO'}")

    # ── Test 3: Residual speedup > Spatial speedup? ──
    # Pair by (n_samples, seed)
    merged = df[df['criterion'] == 'Residual'][['n_samples', 'seed', 'speedup']].merge(
        df[df['criterion'] == 'Spatial'][['n_samples', 'seed', 'speedup']],
        on=['n_samples', 'seed'],
        suffixes=('_res', '_spa')
    )
    diffs = merged['speedup_res'].values - merged['speedup_spa'].values
    stat3, p3 = wilcoxon(diffs, alternative='greater')
    results['residual_faster_than_spatial'] = {'statistic': stat3, 'p_value': p3}
    print(f"\n[Test 3] Residual speedup > Spatial speedup?")
    print(f"  W={stat3:.2f}, p={p3:.4f} → {'✅ YES (p<0.05)' if p3 < 0.05 else '❌ NO'}")
    print(f"  Mean difference: {diffs.mean():.2f}x (residual - spatial)")

    return pd.DataFrame(results).T.reset_index().rename(
        columns={'index': 'test'})


# ── Pretty print for paper ─────────────────────────────────
def print_latex_summary(summary):
    print("\n" + "="*60)
    print("  LATEX TABLE (mean ± std, 5 runs)")
    print("="*60)

    for crit in ['Residual', 'Spatial']:
        sub = summary[summary['criterion'] == crit].sort_values('n_samples')
        print(f"\n% --- {crit} ---")
        for _, r in sub.iterrows():
            n   = int(r['n_samples'])
            sp  = f"{r['speedup_mean']:.1f} \\pm {r['speedup_std']:.1f}"
            r2  = f"{r['r2_sub_mean']:.3f} \\pm {r['r2_sub_std']:.3f}"
            t   = f"{r['time_sub_mean']:.1f} \\pm {r['time_sub_std']:.1f}"
            print(f"  {n:>6,} & {sp} & {r2} & {t} \\\\")


# ── Entry point ────────────────────────────────────────────
if __name__ == '__main__':
    if not SUBSAMPLE_AVAILABLE:
        print("Cannot run: svr_residual_subsample not found.")
        exit(1)

    print("="*60)
    print("  SVR SUBSAMPLING — STATISTICAL VALIDATION")
    print(f"  Seeds:  {SEEDS}")
    print(f"  Sizes:  {SAMPLE_SIZES}")
    print(f"  Criteria: Residual + Spatial")
    print("="*60)

    # 1. Run all experiments
    df_raw = run_experiment()
    df_raw.to_csv('/content/drive/MyDrive/SVR Project/Stats Test/all_runs_raw.csv', index=False)
    print("\n✅ Raw data saved → all_runs_raw.csv")

    # 2. Summary statistics
    summary = compute_summary(df_raw)
    summary.to_csv('/content/drive/MyDrive/SVR Project/Stats Test/summary_stats.csv', index=False)
    print("✅ Summary stats saved → summary_stats.csv")

    # 3. Wilcoxon tests
    wilcoxon_results = run_wilcoxon_tests(df_raw)
    wilcoxon_results.to_csv('/content/drive/MyDrive/SVR Project/Stats Test/wilcoxon_results.csv', index=False)
    print("✅ Wilcoxon results saved → wilcoxon_results.csv")

    # 4. LaTeX-ready table
    print_latex_summary(summary)

    print("\n🎉 Done! Files generated:")
    print("   - all_runs_raw.csv      (one row per n × seed × criterion)")
    print("   - summary_stats.csv     (mean ± std per n × criterion)")
    print("   - wilcoxon_results.csv  (3 statistical tests)")

  SVR SUBSAMPLING — STATISTICAL VALIDATION
  Seeds:  [42, 123, 456, 789, 1011]
  Sizes:  [1000, 5000, 10000, 15000, 20000, 30000, 40000, 45000, 50000]
  Criteria: Residual + Spatial
  📂 Loaded Traditional SVR checkpoint: [1000, 5000, 10000, 15000, 20000, 30000, 40000, 45000]
  📂 Loaded subsample checkpoint: 80 rows, 80 completed (n, seed, criterion) combos

  n = 1,000
  [Traditional SVR] ✅ loaded from checkpoint — time=9.9s  R²=0.8322
  [Residual] seed=42 ✅ already done — skipping
  [Spatial] seed=42 ✅ already done — skipping
  [Residual] seed=123 ✅ already done — skipping
  [Spatial] seed=123 ✅ already done — skipping
  [Residual] seed=456 ✅ already done — skipping
  [Spatial] seed=456 ✅ already done — skipping
  [Residual] seed=789 ✅ already done — skipping
  [Spatial] seed=789 ✅ already done — skipping
  [Residual] seed=1011 ✅ already done — skipping
  [Spatial] seed=1011 ✅ already done — skipping
  ✅ n=1,000 complete — 80 total rows saved.

  n = 5,000
  [Traditional SVR] ✅ loaded

"Utilizamos el kernel RBF (Radial Basis Function) ya que el dataset Friedman #1 contiene relaciones no lineales complejas que incluyen funciones trigonométricas e interacciones entre variables. El kernel RBF es capaz de mapear estos datos a un espacio de dimensión infinita donde estas relaciones se vuelven linealmente separables."

"While based on Camelo et al.'s framework, we introduce a
FUNDAMENTAL modification: residual-based neighbor selection
instead of spatial distance. This changes the algorithm from
geometric clustering to error-pattern clustering, with
significant theoretical and practical implications."